# AIDS: intersectional bias analysis

**Goal:** Evaluating dataset viability to explore intersectional bias, finding elligible crossing between sensitive attributes and detecting baseline inbalances.

## Setup

AIDS dataset contains the following sensitive attributes: *homo*, *gender*, *race*, *age*. We need to evaluate how combinations of these experiments generate interesting results for intersectional bias. To view insights on the dataset variables: https://archive.ics.uci.edu/dataset/890/aids+clinical+trials+group+study+175

Some binary definitions in the dataset:

```
race: (0=White, 1=non-white)
gender: (0=F, 1=M)
homo: (0=no, 1=yes)
cid: censoring indicator (1 = failure, 0 = censoring)
```

In [ ]:
import sys 
import os

sys.path.append(os.path.abspath("../utils"))
from config import DATA_PATH

# Target variable = censoring indicator (cid). 1 means death, 0 means censored
# already renamed cid to target in pre-processed table
TARGET_COL = "target"
FAVORABLE_LABEL = 0

# Sensitive attributes to evaluate
SENSITIVE_ATTRS = ["race", "gender", "homo"]

# Privileged subgroups
PRIVILEGED =  {
    "gender": 1, # male
    "race": 0, # white
    "homo": 1 # yes?
}

MIN_SUBGROUP_COUNT = 80

### Visual configurations

In [4]:
import warnings
warnings. filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

import seaborn as sab

from scipy import stats
from scipy.stats import chi2_contingency

LAB_PALETTE_SEQUENTIAL = [
    "#f0f9fa", # Ice white (Almost white, close to no correlation)
    "#b2ebf2", # Acqua
    "#3eccc4", # Teal (signs start to show up)
    "#1a659e", # Light Blue (moderate association)
    "#002a61"  # Navy Blue (critical association)
]

# global seaborn configurations

sab.set_theme(style="whitegrid")
sab.set_palette(LAB_PALETTE_SEQUENTIAL)

# custom heatmap based on palette
LAB_CMAP = LinearSegmentedColormap.from_list("LabGradient", LAB_PALETTE_SEQUENTIAL)

# LAB_PALETTE_20 for high density (20 colors)
LAB_PALETTE_CATEGORICAL = [
    # NAVY (Higher privilege/status)
    "#001b3d", # Navy Ultra Dark
    "#002a61", # Original Navy
    "#004b8d", # Medium Navy
    "#006cc2", # Shiny Navy
    
    # BLUE (Upper intermediate groups)
    "#134e7a", # Deep Blue
    "#1a659e", # Original Blue
    "#4d8fc1", # Steel Blue
    "#82b4dc", # Sky Blue
    
    # PURPLE (Transition/ Intersection zones)
    "#2e2c8f", # Deep Purple
    "#403eb1", # Purple Original
    "#6b69d6", # Medium Purple
    "#8e8cf2", # Lavender 
    
    # TEAL (Parity groups)
    "#168275", # Deep Teal
    "#1eaf9d", # Teal Original
    "#28d1bd", # Bright Teal
    "#5fe3d3", # Mint Teal
    
    # AQUA (Higher vulnerability)
    "#2fa39d", # Dark Aqua
    "#3eccc4", # Aqua Original
    "#66cbe1", # Sky Blue Original
    "#b3b1ff"  # Pale Lavender (lightest point)
]

sab.set_palette(LAB_PALETTE_CATEGORICAL)

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## Functions
The statistic method used for this evaluation is the Wilson Score Interval. It performs better for smaller instance numbers, avoiding noise-generated bias in samples.

In [5]:
def wilson_ci(k, n, z=1.96):
    '''Wilson Score Interval'''
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    spread = (z * np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return float(center - spread), float(center + spread)

def subgroup_label_stats(df, subgroup_col, target_col, favorable_label):
    '''Calculates favorable rate and WCI for each subgroup'''
    records = []
    for sg, grp in df.groupby(subgroup_col):
        n = len(grp)
        k = (grp[target_col] == favorable_label).sum()
        rate = k / n if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)
        records.append({
            "subgroup": sg, "n": n, "favorable_n": int(k),
            "favorable_rate": rate, "IC_low": lo, "IC_high": hi
        })
    return pd.DataFrame(records).sort_values("subgrupo").reset_index(drop=True)

def cramers_v(df, col1, col2):
    '''Cramer's V - determines how strongly two attributes are associated'''
    ct = pd.crosstab(df[col1], df[col2])
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0

print("✓ Done importing functions")

✓ Done importing functions


## Pre-Processing
A bit of pre-processing is already executed during the `init_datasets` notebook. This section takes care of any extra needed processing and visualization

In [13]:
processed_data_path = DATA_PATH / "processed"
df_pp = pd.read_csv(processed_data_path / f"6_aids_pp.csv")
# creating age-group category
bins = [0, 18, 34, 50, 65, 80]
labels = ['0-18', '19-34', '35-50', '51-65', '66-80']
df_pp['age_group'] = pd.cut(df_pp['age'], bins=bins, labels=labels, right=True)
df_pp.head(5)

,trt,age,wtkg,hemo,homo,drugs,karnof,oprior,z30,zprior,...,strat,symptom,treat,offtrt,cd40,cd420,cd80,cd820,target,age_group
0,2,48,89.8128,0,0,0,100,0,0,1,...,1,0,1,0,422,477,566,324,0,35-50
1,3,61,49.4424,0,0,0,90,0,1,1,...,3,0,1,0,162,218,392,564,1,51-65
2,3,45,88.4520,0,1,1,90,0,1,1,...,3,0,1,1,326,274,2063,1893,0,35-50
3,3,47,85.2768,0,1,0,100,0,1,1,...,3,0,1,0,287,394,1590,966,0,35-50
4,0,43,66.6792,0,1,0,100,0,1,1,...,3,0,0,0,504,353,870,782,0,35-50


## Sensitive Attributes Inventory

In [14]:
SECONDARY_ATTRS = ["age_group"]
ALL_ANALYSIS_ATTRS = SENSITIVE_ATTRS + SECONDARY_ATTRS

for attr in ALL_ANALYSIS_ATTRS:
    print(f"  {attr} (type: {df_pp[attr].dtype})")

    # Calculating frequencies and proportions
    counts = df_pp[attr].value_counts().sort_index()
    percents = df_pp[attr].value_counts(normalize=True).sort_index() * 100

    # Formatting view
    for val, count in counts.items():
        pct = percents[val]
        print(f"    {str(val):<30} n={count:>6,}  ({pct:>5.1f}%)")
    print()

# Checking for null values
null_check = df_pp[ALL_ANALYSIS_ATTRS].isnull().sum().sum()
if null_check == 0:
    print(f"✓ Quality Check: No missing values in analysis attributes.")
else:
    print(f"⚠ Warning: Found {null_check} missing values.")


  race (type: int64)
    0                              n= 1,522  ( 71.2%)
    1                              n=   617  ( 28.8%)

  gender (type: int64)
    0                              n=   368  ( 17.2%)
    1                              n= 1,771  ( 82.8%)

  homo (type: int64)
    0                              n=   725  ( 33.9%)
    1                              n= 1,414  ( 66.1%)

  age_group (type: category)
    0-18                           n=    33  (  1.5%)
    19-34                          n= 1,053  ( 49.2%)
    35-50                          n=   950  ( 44.4%)
    51-65                          n=    95  (  4.4%)
    66-80                          n=     8  (  0.4%)

✓ Quality Check: No missing values in analysis attributes.
